# Execute production

**LD-BT model-optimization step.** Train and validate the two-stage active-learning model on collected DBTL data. Stage 1 trains a classifier (active vs inactive); stage 2 trains a regressor on activity, applied only to predicted-active variants. For each (model, scoring) combination the pipeline runs `RandomizedSearchCV` and writes CV + validation tables to the output directory.

**Data requirement.** Place the campaign's collected-activity workbook at `data/<name>/training_activity_data.xlsx` (and, for `aaindex`/`esmfold` encodings, the corresponding feature files in the same folder). The variant-library xlsx shipped in `data/` is the public summary; the per-residue training workbook is the model input and is not redistributed here.

**Runtime.** `N_ITER = 100` across model families is a multi-hour run. For a quick check, reduce `N_ITER` and/or the model subsets in the Parameters cell.

**Figures:** two-stage model validation (Fig 3, Fig 4; UGTSL2 transfer Fig 7).

In [ ]:
import os, sys
ROOT = os.path.abspath(os.getcwd())
SRC = os.path.join(ROOT, 'src')
if SRC not in sys.path:
    sys.path.insert(0, SRC)
print('repo root :', ROOT)
print('src on path:', SRC in sys.path)

In [ ]:
from ml.data_loading import load_and_prepare, split_train_val, COL_CONVERSION, COL_ACTIVITY
from ml.feature_engineering import get_features
from ml.models import CLASSIFIERS, REGRESSORS
from ml.evaluation import run_classification, run_regression
print('pipeline loaded; classifiers =', list(CLASSIFIERS))
print('regressors  =', list(REGRESSORS))

## Parameters
Defaults reproduce the published one-hot (partial-scope) run.

In [ ]:
NAME             = 'SrUGT76G1'
ENCODING         = 'one_hot'          # one_hot | protparams | aaindex | esmfold
SCOPE            = 'partial'          # partial | full
STAGE            = 'both'             # classification | regression | both
OBJ_COL          = COL_ACTIVITY       # regression target column
TRAIN_FILTER     = 'all'             # all | pos_only
VAL_DATA         = 'l2'              # l2 | l2_predconv
CLF_METRICS      = ['average_precision']
REG_METRICS      = ['spearman', 'ndcg']
CLASSIFIER_NAMES = list(CLASSIFIERS)  # subset to run
REGRESSOR_NAMES  = list(REGRESSORS)
N_ITER           = 100                # paper used 100; reduce for a quick run
OUTPUT_PATH      = os.path.join('results', NAME + '_production')
DATA_DIR         = os.path.join('data', NAME)
print('encoding :', ENCODING, '(' + SCOPE + ')')
print('stage    :', STAGE)
print('data dir :', DATA_DIR)

## Check training data is present

In [ ]:
train_xlsx = os.path.join(DATA_DIR, 'training_activity_data.xlsx')
if not os.path.exists(train_xlsx):
    raise FileNotFoundError(
        'Training workbook not found at %s.\n'
        'This per-residue input is not redistributed; see the markdown above.' % train_xlsx)
os.makedirs(OUTPUT_PATH, exist_ok=True)
print('found training data; outputs ->', OUTPUT_PATH)

## Load and split

In [ ]:
df = load_and_prepare(DATA_DIR)
df_train, df_val = split_train_val(df, train_filter=TRAIN_FILTER,
                                   val_data=VAL_DATA, data_dir=DATA_DIR)
print('train rows :', len(df_train), '| val rows :', len(df_val))

## Featurize

In [ ]:
X_train = get_features(df_train, ENCODING, SCOPE, data_dir=DATA_DIR)
X_val   = get_features(df_val,   ENCODING, SCOPE, data_dir=DATA_DIR)
scope_label = ENCODING + '_' + SCOPE
print('X_train :', X_train.shape, '| X_val :', X_val.shape)

## Stage 1 - classification

In [ ]:
if STAGE in ('classification', 'both'):
    clf = {k: CLASSIFIERS[k] for k in CLASSIFIER_NAMES if k in CLASSIFIERS}
    run_classification(X_train, df_train[COL_CONVERSION], X_val, df_val[COL_CONVERSION],
                       clf, CLF_METRICS, ENCODING, scope_label,
                       TRAIN_FILTER, VAL_DATA, OUTPUT_PATH, n_iter=N_ITER)
    print('classification done')
else:
    print('classification skipped (STAGE =', STAGE, ')')

## Stage 2 - regression

In [ ]:
if STAGE in ('regression', 'both'):
    reg = {k: REGRESSORS[k] for k in REGRESSOR_NAMES if k in REGRESSORS}
    run_regression(X_train, df_train[OBJ_COL], X_val, df_val[OBJ_COL],
                   reg, REG_METRICS, ENCODING, scope_label,
                   TRAIN_FILTER, VAL_DATA, OUTPUT_PATH, n_iter=N_ITER)
    print('regression done')
else:
    print('regression skipped (STAGE =', STAGE, ')')

## Two-stage cascade
Compose the cascade from the validation tables written above: the **top-ranked** classifier and regressor are selected automatically. To pick a specific model, set `CLF_MODEL` / `REG_MODEL` to a name in `CLASSIFIERS` / `REGRESSORS` (leave `None` for the top-ranked one). The classifier calls active/inactive; the regressor ranks **only** the predicted-active variants (predicted-inactive rank last). This is the navigation / rank-correlation predictor (Fig 4).

The demo uses the notebook's single `ENCODING` for both stages; the published pipeline uses an ESM-Partial classifier and a ProtParam regressor - build each with `get_features(...)` and pass them as `X_clf` / `X_reg`.

In [ ]:
from ml.two_stage import rank_models, select_model, fit_stage_model, two_stage_predict

CLF_MODEL = None   # name in CLASSIFIERS to pick one; None -> top-ranked
REG_MODEL = None   # name in REGRESSORS to pick one; None -> top-ranked

print('classifier ranking:'); print(rank_models(OUTPUT_PATH, 'classification').to_string(index=False))
print('\nregressor ranking:'); print(rank_models(OUTPUT_PATH, 'regression').to_string(index=False))

clf_name = select_model(OUTPUT_PATH, 'classification', model_name=CLF_MODEL)
reg_name = select_model(OUTPUT_PATH, 'regression', model_name=REG_MODEL)
print('\ncascade: classifier=%s, regressor=%s' % (clf_name, reg_name))

clf = fit_stage_model(OUTPUT_PATH, 'classification', clf_name, X_train, df_train[COL_CONVERSION])
active_tr = (df_train[COL_CONVERSION] == 1).to_numpy()   # regressor trains on actives only
reg = fit_stage_model(OUTPUT_PATH, 'regression', reg_name, X_train[active_tr], df_train.loc[active_tr, OBJ_COL])

cascade = two_stage_predict(clf, reg, X_val, X_val, ids=df_val['ID'].values)
print('\npredicted active in validation: %d / %d' % (cascade['predicted_active'].sum(), len(cascade)))
cascade.sort_values('rank').head(10)

In [ ]:
print('done. tables written under', OUTPUT_PATH)